# Redrob Intelligent Candidate Ranker — Sandbox Demo

**Team:** Kittu  
**Challenge:** India Runs Data & AI Challenge — Senior AI Engineer JD

This notebook demonstrates the ranker running on the 50-candidate sample file included in the repo.
The full submission was produced on 100,000 candidates in ~40 seconds on CPU.

---
### Approach
Multi-signal rule-based ranker with 5 weighted components:
- **Skill Match (35%)** — Tier A/B/C AI keyword matching with proficiency & duration weighting
- **Title/Headline (20%)** — Keyword + summary semantic scoring
- **Career Depth (15%)** — YoE fit curve, startup experience, production ML signals in descriptions
- **Education (5%)** — Institution tier + field relevance
- **Location (5%)** — India Tier-1 cities preferred
- **Behavioral Multiplier (×0.1–1.15)** — Recency, open-to-work, response rate, GitHub, notice period

Plus honeypot detection (10 checks for impossible profiles).

## Step 1: Clone the Repository

In [ ]:
!git clone https://github.com/Kritansh-Tank/redrob-ranker.git
%cd redrob-ranker

## Step 2: Install Dependencies

In [ ]:
!pip install -r requirements.txt -q
print('Dependencies installed.')

## Step 3: Run the Ranker on Sample Candidates

The repo includes `sample_candidates.json` — 50 candidates from the challenge dataset.  
The `--top 50` flag outputs rankings for all available candidates (instead of defaulting to 100).

In [ ]:
!python rank.py \
    --candidates ./sample_candidates.json \
    --out ./sample_output.csv \
    --top 50 \
    --preview

## Step 4: View Full Results

In [ ]:
import pandas as pd

df = pd.read_csv('sample_output.csv')
print(f'Total ranked: {len(df)} candidates\n')
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_rows', 50)
df

## Step 5: Score Distribution

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(df['rank'], df['score'], color='steelblue', alpha=0.8)
axes[0].set_xlabel('Rank')
axes[0].set_ylabel('Score')
axes[0].set_title('Score vs Rank')
axes[0].grid(axis='y', alpha=0.3)

axes[1].hist(df['score'], bins=15, color='darkorange', alpha=0.8, edgecolor='white')
axes[1].set_xlabel('Score')
axes[1].set_ylabel('Count')
axes[1].set_title('Score Distribution')
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('Redrob Candidate Ranker — Sample Results (50 candidates)', fontsize=13)
plt.tight_layout()
plt.show()

print(f"Top score:  {df['score'].max():.4f}")
print(f"Min score:  {df['score'].min():.4f}")
print(f"Mean score: {df['score'].mean():.4f}")

## Step 6: Inspect Top 5 Candidates

In [ ]:
import json

with open('sample_candidates.json') as f:
    all_cands = {c['candidate_id']: c for c in json.load(f)}

top5 = df.head(5)
for _, row in top5.iterrows():
    c = all_cands.get(row['candidate_id'], {})
    p = c.get('profile', {})
    sig = c.get('redrob_signals', {})
    skills = [s['name'] for s in c.get('skills', [])[:6]]
    print(f"Rank {int(row['rank'])} | {row['candidate_id']} | Score {row['score']}")
    print(f"  Title    : {p.get('current_title', '?')} | {p.get('years_of_experience', '?')}y exp")
    print(f"  Location : {p.get('location', '?')}, {p.get('country', '?')}")
    print(f"  Skills   : {', '.join(skills)}")
    print(f"  GitHub   : {sig.get('github_activity_score', -1)} | Response rate: {sig.get('recruiter_response_rate', 0):.0%} | Notice: {sig.get('notice_period_days', '?')}d")
    print(f"  Reasoning: {row['reasoning'][:110]}")
    print()

---
**Repo:** https://github.com/Kritansh-Tank/redrob-ranker  
**Full run command (100K candidates):**
```bash
python rank.py --candidates ./candidates.jsonl --out ./submission.csv
```
Runtime: ~40 seconds | CPU only | No external APIs